# 🌐 InvestAI — API REST (FastAPI + MongoDB + ngrok)

**Proyecto:** Sistema Ernesto Investing AI  
**Módulo:** API REST de solo lectura sobre datos pre-calculados  
**Fuente de datos:** MongoDB Atlas (`precios_ohlcv`, `predicciones`)  
**Despliegue:** Google Colab + ngrok (túnel público)  

---

A diferencia de los notebooks de Ingesta y Clasificación (que escriben
datos), este notebook implementa una **API REST de solo lectura**: no
recalcula nada, simplemente expone vía HTTP los datos que ya fueron
procesados y almacenados en MongoDB por los módulos anteriores.

### Endpoints expuestos

| Método | Endpoint | Descripción | Colección leída |
|---|---|---|---|
| `GET` | `/api/salud` | Estado de salud del servidor | — |
| `GET` | `/api/mercado/{ticker}` | Serie OHLCV + indicadores técnicos | `precios_ohlcv` |
| `GET` | `/api/svc/{ticker}` | Señal vigente del clasificador SVC | `predicciones` |

> ⚠️ **Requisitos previos:**
> - `MONGO_URI` configurado en los Secrets de Colab.
> - `NGROK_AUTHTOKEN` configurado en los Secrets de Colab.
> - Haber ejecutado los notebooks de Ingesta y Clasificación SVC al
>   menos una vez (para que existan datos que leer).

## Sección 1 — Instalación de Dependencias

In [ ]:
# fastapi + uvicorn   : framework de API REST y servidor ASGI
# pyngrok             : túnel público hacia el puerto local de Colab
# pymongo[srv]        : cliente de MongoDB con soporte Atlas (SRV)
# nest-asyncio        : permite ejecutar el loop de uvicorn dentro del loop de Colab
!pip install fastapi uvicorn pyngrok "pymongo[srv]" nest-asyncio --quiet

print("✅ Dependencias instaladas correctamente.")

## Sección 2 — Importación de Librerías

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import threading
import time
from datetime import datetime
from typing import Optional, Any, Dict, List

import nest_asyncio
import uvicorn

from fastapi import FastAPI, HTTPException, Path
from fastapi.middleware.cors import CORSMiddleware

from pymongo import MongoClient
from pymongo.errors import ConnectionFailure

from pyngrok import ngrok, conf
from pyngrok.exception import PyngrokNgrokError

from google.colab import userdata

print("✅ Librerías importadas correctamente.")

## Sección 3 — Configuración Global

In [ ]:
# ─── Tickers del estudio (para validar parámetros de ruta) ────────────────
TICKERS_VALIDOS = ['FSM', 'VOLCABC1.LM', 'ABX.TO', 'BVN', 'BHP']

# ─── Configuración de MongoDB ──────────────────────────────────────────────
DB_NOMBRE        = 'investai'
COL_PRECIOS      = 'precios_ohlcv'
COL_PREDICCIONES = 'predicciones'

# ─── Configuración del servidor ────────────────────────────────────────────
PUERTO_LOCAL = 8000

# Campos a excluir SIEMPRE de las respuestas HTTP (no son útiles para el
# frontend y exponen detalles internos de almacenamiento)
CAMPOS_EXCLUIDOS = {'_id': 0, 'created_at': 0}

print("✅ Configuración cargada.")
print(f"   Tickers válidos : {TICKERS_VALIDOS}")
print(f"   Base de datos   : {DB_NOMBRE}")
print(f"   Puerto local    : {PUERTO_LOCAL}")

## Sección 4 — Conexión a MongoDB Atlas

La conexión se crea **una sola vez** al iniciar el notebook y se
reutiliza en todas las peticiones HTTP (patrón recomendado de pymongo:
el `MongoClient` ya gestiona internamente un *connection pool*).

In [ ]:
try:
    MONGO_URI = userdata.get('MONGO_URI')
    if not MONGO_URI:
        raise ValueError("El secret MONGO_URI está vacío.")
except Exception as e:
    raise RuntimeError(
        f"No se encontró el secret MONGO_URI: {e}\n"
        "Agrega tu URI en: Panel izquierdo → 🔑 → Agregar nuevo secreto → MONGO_URI"
    )

try:
    cliente_mongo = MongoClient(
        MONGO_URI,
        serverSelectionTimeoutMS=10_000,
        connectTimeoutMS=10_000,
    )
    cliente_mongo.admin.command('ping')
    print("✅ Conexión a MongoDB Atlas establecida correctamente.")
except ConnectionFailure as e:
    raise RuntimeError(
        f"No se pudo conectar a MongoDB Atlas: {e}\n"
        "Verifica que la URI sea correcta y que la IP de Colab esté en la lista blanca de Atlas."
    )

db                = cliente_mongo[DB_NOMBRE]
col_precios       = db[COL_PRECIOS]
col_predicciones  = db[COL_PREDICCIONES]

n_precios      = col_precios.count_documents({})
n_predicciones = col_predicciones.count_documents({})

print(f"   Colección '{COL_PRECIOS}'      : {n_precios:,} documentos")
print(f"   Colección '{COL_PREDICCIONES}'       : {n_predicciones:,} documentos")

if n_precios == 0:
    print("\n⚠️  ADVERTENCIA: 'precios_ohlcv' está vacía. Ejecuta primero el notebook de Ingesta.")
if n_predicciones == 0:
    print("⚠️  ADVERTENCIA: 'predicciones' está vacía. Ejecuta primero el notebook de Clasificación SVC.")

## Sección 5 — Configuración del Túnel ngrok

El *authtoken* de ngrok se lee desde los **Secrets de Colab**
(`NGROK_AUTHTOKEN`) para no exponer credenciales en el código.

> En `pyngrok` 8.x, el authtoken se asigna mediante
> `conf.get_default().auth_token` (en versiones anteriores se usaba
> `ngrok.set_auth_token()`, que sigue funcionando pero internamente
> delega a este mismo mecanismo).

In [ ]:
try:
    NGROK_AUTHTOKEN = userdata.get('NGROK_AUTHTOKEN')
    if not NGROK_AUTHTOKEN:
        raise ValueError("El secret NGROK_AUTHTOKEN está vacío.")
except Exception as e:
    raise RuntimeError(
        f"No se encontró el secret NGROK_AUTHTOKEN: {e}\n"
        "1. Crea una cuenta gratuita en https://dashboard.ngrok.com/signup\n"
        "2. Copia tu authtoken desde https://dashboard.ngrok.com/get-started/your-authtoken\n"
        "3. Agrégalo en: Panel izquierdo → 🔑 → Agregar nuevo secreto → NGROK_AUTHTOKEN"
    )

# pyngrok 8.x: asignación del authtoken vía el objeto de configuración global
conf.get_default().auth_token = NGROK_AUTHTOKEN

# Cierra túneles previos por si la celda se ejecuta más de una vez en la misma sesión
ngrok.kill()

print("✅ Token de ngrok configurado correctamente (pyngrok 8.x).")

## Sección 6 — Definición de la Aplicación FastAPI

Aplicación de **solo lectura**: cada endpoint hace una consulta
directa a MongoDB y devuelve el resultado tal cual está almacenado
(excluyendo `_id` y `created_at`), sin recalcular nada.

In [ ]:
app = FastAPI(
    title="InvestAI API",
    description=(
        "API REST de solo lectura del Sistema InvestAI. Expone datos "
        "de mercado e indicadores técnicos, y señales del clasificador "
        "SVC, previamente calculados y almacenados en MongoDB Atlas "
        "por los módulos de Ingesta y Clasificación."
    ),
    version="1.0.0",
    docs_url="/docs",
    redoc_url="/redoc",
)

# CORS habilitado para cualquier origen — permite que el frontend
# (servido desde otro dominio/archivo local) consuma la API sin bloqueos
app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

print("✅ Aplicación FastAPI creada con CORS habilitado (allow_origins=['*']).")

In [ ]:
@app.get("/api/salud", tags=["Salud"], summary="Estado de salud del servidor")
def api_salud():
    """
    Health check del servidor: confirma que la API está activa y que
    la conexión a MongoDB funciona, reportando el conteo de documentos
    disponibles en cada colección.
    """
    try:
        # Verificar conectividad en tiempo real (no solo al iniciar el notebook)
        cliente_mongo.admin.command('ping')
        mongo_ok = True
    except Exception:
        mongo_ok = False

    return {
        'status': 'ok' if mongo_ok else 'degradado',
        'servicio': 'InvestAI API',
        'version': '1.0.0',
        'timestamp': datetime.now().isoformat(),
        'mongodb_conectado': mongo_ok,
        'documentos_precios_ohlcv': col_precios.count_documents({}) if mongo_ok else None,
        'documentos_predicciones': col_predicciones.count_documents({}) if mongo_ok else None,
        'tickers_disponibles': TICKERS_VALIDOS,
    }

print("✅ Endpoint GET /api/salud definido.")

In [ ]:
@app.get("/api/mercado/{ticker}", tags=["Mercado"],
         summary="Serie OHLCV con indicadores técnicos (lectura desde MongoDB)")
def api_mercado(ticker: str = Path(..., description="Símbolo bursátil, ej. 'BVN'")):
    """
    Devuelve la serie completa de precios OHLCV + indicadores técnicos
    (SMA-20, SMA-50, EMA-12, EMA-26, RSI-14) almacenada previamente en
    la colección 'precios_ohlcv' por el módulo de Ingesta de Datos.

    Este endpoint NO descarga datos de Yahoo Finance ni recalcula
    indicadores: únicamente lee lo que ya existe en MongoDB.
    """
    ticker = ticker.strip().upper()

    # Consulta a MongoDB: todos los documentos del ticker, ordenados por fecha,
    # excluyendo los campos internos _id y created_at
    cursor = col_precios.find(
        {'ticker': ticker},
        CAMPOS_EXCLUIDOS
    ).sort('fecha', 1)

    serie = list(cursor)

    if not serie:
        raise HTTPException(
            status_code=404,
            detail=(
                f"No hay datos almacenados para el ticker '{ticker}' en "
                f"'{COL_PRECIOS}'. Verifica el símbolo o ejecuta primero "
                f"el notebook de Ingesta de Datos."
            ),
        )

    # Convertir datetimes a string ISO para que sean serializables por FastAPI
    for doc in serie:
        if isinstance(doc.get('fecha'), datetime):
            doc['fecha'] = doc['fecha'].isoformat()
        if isinstance(doc.get('ingestado_en'), datetime):
            doc['ingestado_en'] = doc['ingestado_en'].isoformat()

    ultimo = serie[-1]

    return {
        'ticker': ticker,
        'nombre': ultimo.get('nombre', ticker),
        'moneda': ultimo.get('moneda', 'USD'),
        'total_registros': len(serie),
        'precio_actual': ultimo.get('cierre'),
        'fecha_actual': ultimo.get('fecha_str') or ultimo.get('fecha'),
        'serie': serie,
    }

print("✅ Endpoint GET /api/mercado/{ticker} definido.")

In [ ]:
@app.get("/api/svc/{ticker}", tags=["Modelos IA"],
         summary="Señal vigente del clasificador SVC (lectura desde MongoDB)")
def api_svc(ticker: str = Path(..., description="Símbolo bursátil, ej. 'BVN'")):
    """
    Devuelve la señal de inversión (BUY/SELL) más reciente generada
    por el clasificador SVC, junto con su nivel de confianza,
    almacenada previamente en la colección 'predicciones' por el
    módulo de Clasificación SVC.

    Este endpoint NO entrena ni reentrena ningún modelo: únicamente
    lee la predicción ya calculada y persistida en MongoDB.
    """
    ticker = ticker.strip().upper()

    doc = col_predicciones.find_one(
        {'ticker': ticker, 'modelo': 'SVC'},
        CAMPOS_EXCLUIDOS
    )

    if not doc:
        raise HTTPException(
            status_code=404,
            detail=(
                f"No hay predicción SVC almacenada para el ticker '{ticker}' "
                f"en '{COL_PREDICCIONES}'. Verifica el símbolo o ejecuta primero "
                f"el notebook de Clasificación SVC."
            ),
        )

    # Convertir datetime a string ISO para que sea serializable por FastAPI
    if isinstance(doc.get('actualizado_en'), datetime):
        doc['actualizado_en'] = doc['actualizado_en'].isoformat()

    return doc

print("✅ Endpoint GET /api/svc/{ticker} definido.")

In [ ]:
@app.get("/", tags=["Salud"], include_in_schema=False)
def raiz():
    """Ruta raíz: mensaje de bienvenida con referencia a la documentación."""
    return {
        'mensaje': 'InvestAI API — visite /docs para la documentación interactiva (Swagger UI)',
        'endpoints': ['/api/salud', '/api/mercado/{ticker}', '/api/svc/{ticker}'],
    }

print("✅ Ruta raíz '/' definida.")
print("\n📋 Resumen de endpoints registrados en la app FastAPI:")
for ruta in app.routes:
    if hasattr(ruta, 'methods'):
        print(f"   {list(ruta.methods)[0]:<6} {ruta.path}")

## Sección 7 — Despliegue: Túnel ngrok + Servidor uvicorn

Esta celda:
1. Abre un túnel público de ngrok hacia el puerto `8000`.
2. Levanta el servidor `uvicorn` en un **hilo separado** (para no
   bloquear el resto del notebook) usando `nest_asyncio`.
3. Mantiene la celda activa mientras el servidor esté corriendo.

⚠️ La celda queda **ejecutándose de forma bloqueante** mientras el
servidor está activo (esto es esperado). Para detenerlo, interrumpe
la ejecución de la celda (■ *Stop*).

In [ ]:
# nest_asyncio permite ejecutar un servidor asyncio (uvicorn) dentro
# del bucle de eventos que ya usa el propio notebook de Colab
nest_asyncio.apply()

url_publica   = None
server_thread = None


def ejecutar_servidor_uvicorn(app_instancia, host: str, puerto: int):
    """Ejecuta el servidor uvicorn de forma bloqueante (pensado para hilo separado)."""
    uvicorn.run(app_instancia, host=host, port=puerto, log_level="info")


try:
    # Abrir el túnel público hacia el puerto local del servidor
    tunel = ngrok.connect(PUERTO_LOCAL)
    url_publica = tunel.public_url

    print("=" * 70)
    print("  🌐 InvestAI API — Servidor desplegado")
    print("=" * 70)
    print(f"  API pública      : {url_publica}")
    print(f"  Swagger UI       : {url_publica}/docs")
    print(f"  ReDoc            : {url_publica}/redoc")
    print(f"  Health check     : {url_publica}/api/salud")
    print("=" * 70)
    print("\n🚀 Iniciando servidor uvicorn en el puerto 8000 (hilo separado)...\n")

    # Iniciar uvicorn en un hilo separado para no bloquear el loop principal de Colab
    server_thread = threading.Thread(
        target=ejecutar_servidor_uvicorn,
        args=(app, "0.0.0.0", PUERTO_LOCAL),
        daemon=True,  # Permite que el proceso termine aunque el hilo siga vivo
    )
    server_thread.start()

    print("✅ Servidor activo. Copia la URL pública de arriba para usarla en el frontend.")
    print("   Presiona el botón ⏹ Stop de esta celda para detener el servidor y cerrar ngrok.\n")

    # Mantener la celda viva mientras el hilo del servidor esté corriendo
    while server_thread.is_alive():
        time.sleep(1)

except PyngrokNgrokError as e:
    print(f"❌ Error al conectar con ngrok: {e}")
    print("   Verifica que tu NGROK_AUTHTOKEN sea válido y que el servicio esté disponible.")
except KeyboardInterrupt:
    print("\n🚨 Servidor interrumpido manualmente.")
except Exception as e:
    print(f"❌ Error inesperado: {e}")
    import traceback
    traceback.print_exc()
finally:
    if url_publica:
        print("\n🛑 Cerrando túnel ngrok...")
        ngrok.disconnect(url_publica)
        print("✅ Túnel ngrok cerrado.")

## Sección 8 — Cómo Probar la API

Mientras la celda anterior siga ejecutándose, el servidor estará
activo. Con la URL pública impresa arriba puedes:

- Abrir `{url_publica}/docs` en el navegador para usar **Swagger UI**
  de forma interactiva.
- Hacer peticiones desde la terminal:

```bash
curl "https://xxxxx.ngrok-free.app/api/salud"
curl "https://xxxxx.ngrok-free.app/api/mercado/BVN"
curl "https://xxxxx.ngrok-free.app/api/svc/BVN"
```

- Conectarte desde el frontend (HTML/JS) gracias a que **CORS** está
  habilitado para cualquier origen.

### Notas

- Esta API es de **solo lectura**: si necesitas datos actualizados,
  vuelve a ejecutar los notebooks de Ingesta y/o Clasificación SVC
  (en otra sesión de Colab, o reiniciando este mismo runtime).
- Si reinicias el túnel ngrok (plan gratuito), la URL pública cambia;
  recuerda actualizarla en el frontend.